In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/processed/cleaned_deliveries.csv",
    dtype={"ball": str, "match_id": str},
    low_memory=False,
)

# Runs the bowler is charged with:
#   runs_off_bat + wides + no-balls + penalty   (NOT byes/leg-byes — those are keeper's)
df["runs_conceded"] = (
    df["runs_off_bat"]
    + df["wides"].fillna(0)
    + df["noballs"].fillna(0)
    + df["penalty"].fillna(0)
)

# Ball counts as legal already exists (df["is_legal"]) — we reuse it.
# Dot ball for bowler = legal ball AND zero runs charged
df["is_dot_for_bowler"] = (df["is_legal"] & (df["runs_conceded"] == 0)).astype(int)

# Bowler wicket already exists (df["is_bowler_wicket"]) — reuse.

print("Rows:", len(df))
print("Unique bowlers:", df["bowler"].nunique())
print("Total legal balls:", int(df["is_legal"].sum()))
print("Total runs conceded:", int(df["runs_conceded"].sum()))
print("Total bowler wickets:", int(df["is_bowler_wicket"].sum()))
print("Total dot balls (bowler view):", int(df["is_dot_for_bowler"].sum()))

Rows: 295557
Unique bowlers: 578
Total legal balls: 284465
Total runs conceded: 394291
Total bowler wickets: 13460
Total dot balls (bowler view): 106302


In [2]:
MIN_BALLS_BOWLING = 300

bowling_career = df.groupby("bowler").agg(
    legal_balls=("is_legal", "sum"),
    runs_conceded=("runs_conceded", "sum"),
    wickets=("is_bowler_wicket", "sum"),
    dot_balls=("is_dot_for_bowler", "sum"),
)

# Innings bowled
bowling_career["innings"] = (
    df[["bowler", "match_id", "innings"]]
    .drop_duplicates()
    .groupby("bowler").size()
)

# Derived metrics (note: economy uses legal balls / 6 for overs)
bowling_career["overs"] = (bowling_career["legal_balls"] / 6).round(1)
bowling_career["economy"]   = (bowling_career["runs_conceded"] / bowling_career["overs"]).round(2)
bowling_career["bowl_sr"]   = np.where(bowling_career["wickets"] > 0,
                                       bowling_career["legal_balls"] / bowling_career["wickets"], np.nan).round(1)
bowling_career["bowl_avg"]  = np.where(bowling_career["wickets"] > 0,
                                       bowling_career["runs_conceded"] / bowling_career["wickets"], np.nan).round(1)
bowling_career["dot_pct"]   = (bowling_career["dot_balls"] / bowling_career["legal_balls"] * 100).round(1)
bowling_career["wickets_per_match"] = (
    bowling_career["wickets"] / bowling_career["innings"]
).round(2)

bowling_career = bowling_career.reset_index().query("legal_balls >= @MIN_BALLS_BOWLING").copy()

print(f"Qualified bowlers (>= {MIN_BALLS_BOWLING} legal balls):", len(bowling_career))
print("\nTop 10 by wickets:")
print(
    bowling_career.nlargest(10, "wickets")[
        ["bowler", "innings", "overs", "wickets", "economy",
         "bowl_sr", "bowl_avg", "dot_pct"]
    ].round(2).to_string(index=False)
)

Qualified bowlers (>= 300 legal balls): 211

Top 10 by wickets:
     bowler  innings  overs  wickets  economy  bowl_sr  bowl_avg  dot_pct
  YS Chahal      184  672.8      233     8.05     17.3      23.2     34.1
    B Kumar      206  766.7      226     7.71     20.4      26.2     43.6
  SP Narine      200  775.2      207     6.79     22.5      25.4     39.9
  PP Chawla      191  641.7      192     7.96     20.1      26.6     35.3
  JJ Bumrah      158  605.2      187     7.34     19.4      23.7     42.2
   R Ashwin      217  785.0      187     7.20     25.2      30.2     35.3
   DJ Bravo      158  520.0      183     8.38     17.0      23.8     33.8
  RA Jadeja      237  709.0      180     7.71     23.6      30.4     32.7
Rashid Khan      153  588.3      179     7.28     19.7      23.9     37.3
   A Mishra      162  561.8      174     7.38     19.4      23.8     36.2


In [3]:
phase_bowl = df.groupby(["bowler", "phase"]).agg(
    legal_balls=("is_legal", "sum"),
    runs_conceded=("runs_conceded", "sum"),
    wickets=("is_bowler_wicket", "sum"),
    dot_balls=("is_dot_for_bowler", "sum"),
).reset_index()

phase_bowl_wide = phase_bowl.pivot(
    index="bowler", columns="phase",
    values=["legal_balls", "runs_conceded", "wickets", "dot_balls"]
)
phase_bowl_wide.columns = [f"{stat}_{ph}" for stat, ph in phase_bowl_wide.columns]
phase_bowl_wide = phase_bowl_wide.reset_index().fillna(0)

for ph in ["Powerplay", "Middle", "Death"]:
    lb  = f"legal_balls_{ph}"
    rn  = f"runs_conceded_{ph}"
    wk  = f"wickets_{ph}"
    dt  = f"dot_balls_{ph}"
    phase_bowl_wide[f"econ_{ph}"] = np.where(
        phase_bowl_wide[lb] > 0,
        (phase_bowl_wide[rn] / (phase_bowl_wide[lb] / 6)).round(2),
        np.nan
    )
    phase_bowl_wide[f"sr_{ph}"] = np.where(
        phase_bowl_wide[wk] > 0,
        (phase_bowl_wide[lb] / phase_bowl_wide[wk]).round(1),
        np.nan
    )
    phase_bowl_wide[f"dot_pct_{ph}"] = np.where(
        phase_bowl_wide[lb] > 0,
        (phase_bowl_wide[dt] / phase_bowl_wide[lb] * 100).round(1),
        np.nan
    )

print("Phase bowling stats built for", len(phase_bowl_wide), "bowlers")
print("\nSample known bowlers:")
sample = phase_bowl_wide[phase_bowl_wide["bowler"].isin(
    ["JJ Bumrah", "SL Malinga", "YS Chahal", "SP Narine", "R Ashwin"]
)][["bowler",
    "legal_balls_Powerplay", "econ_Powerplay",
    "legal_balls_Middle",    "econ_Middle",
    "legal_balls_Death",     "econ_Death",
    "sr_Death"]].round(2)
print(sample.to_string(index=False))

Phase bowling stats built for 578 bowlers

Sample known bowlers:
    bowler  legal_balls_Powerplay  econ_Powerplay  legal_balls_Middle  econ_Middle  legal_balls_Death  econ_Death  sr_Death
 JJ Bumrah                 1266.0            6.94               942.0         6.58             1423.0        8.19      14.0
  R Ashwin                 1252.0            7.09              2857.0         7.03              601.0        8.24      16.7
SL Malinga                 1110.0            6.30               600.0         7.41             1117.0        7.86      10.3
 SP Narine                  990.0            6.93              2556.0         6.53             1105.0        7.25      13.2
 YS Chahal                  404.0            7.87              2960.0         7.75              673.0        9.49      10.2


In [4]:
# Cricsheet stores batter/bowler handedness in the _info.csv files only,
# not in the deliveries CSV. If a batting-hand column exists in df, use it;
# otherwise skip this cell and we'll pull handedness in a later module.
if "batting_hand" in df.columns:
    handedness = df.groupby(["bowler", "batting_hand"]).agg(
        legal_balls=("is_legal", "sum"),
        runs_conceded=("runs_conceded", "sum"),
        wickets=("is_bowler_wicket", "sum"),
    ).reset_index()
    handedness["economy"] = (handedness["runs_conceded"]
                             / (handedness["legal_balls"]/6)).round(2)
    print(handedness.head(6).to_string(index=False))
else:
    print("batting_hand not in df — skipping. "
          "Add later from _info.csv if needed for matchup analysis (Module 5).")

batting_hand not in df — skipping. Add later from _info.csv if needed for matchup analysis (Module 5).


In [5]:
bowling = bowling_career.merge(phase_bowl_wide, on="bowler", how="left")

bowling.to_csv("../data/processed/bowling_stats.csv", index=False)
print("Saved: data/processed/bowling_stats.csv | shape:", bowling.shape)
print()

print("=== Top 10 by economy (min 300 balls) ===")
print(bowling.nsmallest(10, "economy")[
    ["bowler", "innings", "overs", "wickets", "economy", "bowl_sr", "dot_pct"]
].round(2).to_string(index=False))
print()

print("=== Top 10 by dot-ball % (min 300 balls) ===")
print(bowling.nlargest(10, "dot_pct")[
    ["bowler", "innings", "overs", "wickets", "economy", "dot_pct"]
].round(2).to_string(index=False))
print()

print("=== Top 5 death-over specialists (min 100 death balls) ===")
death = bowling.query("legal_balls_Death >= 100").nsmallest(
    5, "econ_Death")[
    ["bowler", "legal_balls_Death", "econ_Death", "sr_Death", "dot_pct_Death"]
].round(2)
print(death.to_string(index=False))

Saved: data/processed/bowling_stats.csv | shape: (211, 33)

=== Top 10 by economy (min 300 balls) ===
          bowler  innings  overs  wickets  economy  bowl_sr  dot_pct
        A Kumble       42  160.8       45     6.58     21.4     41.1
      GD McGrath       14   54.0       12     6.61     27.0     50.0
  M Muralitharan       66  254.0       63     6.68     24.2     42.1
RE van der Merwe       21   73.8       21     6.75     21.1     35.9
      DL Vettori       34  129.5       28     6.79     27.8     36.4
       SP Narine      200  775.2      207     6.79     22.5     39.9
         J Yadav       20   65.0        8     6.85     48.8     39.7
        DW Steyn       95  362.7       97     6.91     22.4     48.9
         J Botha       34  115.7       25     6.91     27.8     38.6
  AD Mascarenhas       13   51.3       19     6.94     16.2     42.2

=== Top 10 by dot-ball % (min 300 balls) ===
       bowler  innings  overs  wickets  economy  dot_pct
   GD McGrath       14   54.0       